# Pattern prediction: RNN vs LSTM

Recurrent networks process sequences one step at a time, maintaining hidden
state across timesteps. This notebook compares two architectures on a binary
pattern prediction task.

- **RNN**: simple recurrence `h' = tanh(W_ih * x + W_hh * h + b)`
- **LSTM**: gated recurrence with forget/input/output gates and cell state

**CLI equivalents:** `make example-rnn` and `make example-lstm`


## Architecture

Both models pair a recurrent cell (1 input feature → 4 hidden) with a
`Linear 4 1` output head. The cells are `Init` builders and implement
`Nn.Recurrent`: `recurStep` consumes the cell (a linear resource), takes one
input, and threads back the output beside the advanced cell. In the compiled
examples the model record mixes multiplicities by role — the stateful `cell`
is a linear field, the read-only `head` is ordinary:

```idris
record Model where
  constructor MkModel
  1 cell : Rnn 1 4 Ex F WithGrad
  head : Linear 4 1 Ex F WithGrad
```


In [1]:
:t rnn

Ml.Nn.Recurrent.rnn : KnownGrad g => Backend ex dt => (TVec o ex dt g' -> IO (TVec o ex dt g')) -> Init (Rnn i o ex dt g)


In [2]:
:t lstm

Ml.Nn.Lstm.lstm : KnownGrad g => Backend ex dt => Init (Lstm i o ex dt g)


In [3]:
:t recurStep

Ml.Nn.Recurrent.recurStep : Recurrent l => Backend ex dt => (1 _ : l i o ex dt WithGrad) -> Tensor [i] ex dt WithGrad -> L IO (LPair ((!*) (Tensor [o] ex dt WithGrad)) (l i o ex dt WithGrad))


## Data: binary pattern prediction

`patternSeqs` (in the examples package, not the kernel prelude) generates 8
sequences of 1D binary values. Given a prefix, the model must predict the next
value at each timestep, so the task tests whether the recurrent model can
learn temporal patterns.


## Training: RNN

The RNN trains with SGD at lr=0.5 under early stopping (`patienceConfig`).
There is no special recurrent driver: the loss function itself folds
`recurStep` over each sequence's timesteps (BCE per step, summed, then
averaged), and `fit` runs that as an ordinary epoch step. From
`Example/Rnn.idr`:

```idris
(MkBang (epochsDone, finalLoss) # trained) <-
  fit (recurEpochL opt) opt (generate (pure patternSeqs)) trainCfg model
```


**Evaluation.** Convert the trained model to inference mode with `eval`
(retypes it `WithGrad -> NoGrad`, runs tape-free), then step it through a
held-out prefix and read the predictions. Tutorial
[05 Model ownership](../tutorials/05_model_ownership.ipynb) covers why the
pre-`eval` handle is unusable afterwards.

## Training: LSTM

The LSTM uses the same task, the same `Linear 4 1` head, and the same
fold-the-loss shape; only the cell builder changes (`lstm {i=1} {o=4}`).
LSTM typically converges faster on sequence tasks thanks to its gating.


## RNN vs LSTM

Key differences:

| | RNN | LSTM |
|---|-----|------|
| Hidden state | Single vector | Cell state + hidden state |
| Gates | None | Forget, input, output |
| Long-range dependencies | Vanishing gradients | Gating preserves information |
| Parameters | Fewer | ~4x more (4 gate matrices) |

For this small task, both converge. On longer sequences (like NTM/DNC tasks),
LSTM's gating becomes essential.

idris-ml also provides `gru` (Gated Recurrent Unit), which has 2 gates
instead of LSTM's 3 — a middle ground between RNN simplicity and LSTM capacity.


In [4]:
:t gru

Ml.Nn.Gru.gru : KnownGrad g => Backend ex dt => Init (Gru i o ex dt g)


## PyTorch comparison

```python
model = nn.RNN(input_size=1, hidden_size=1, batch_first=True)
# or
model = nn.LSTM(input_size=1, hidden_size=1, batch_first=True)

for epoch in range(2000):
    hidden = None
    for x_t, y_t in sequence:
        output, hidden = model(x_t, hidden)
        loss += F.binary_cross_entropy_with_logits(output, y_t)
    loss.backward()
    optimizer.step()
```

In idris-ml the timestep loop is the loss function: `recurStep` threads the
cell through each step, backpropagation through time is ordinary autograd on
the accumulated loss, and `fit` owns the epoch loop, early stopping, and
checkpointing.

See `pytorch/torch_ref/scripts/rnn.py` and `lstm.py` for the full references.


Next: [Transformer](transformer.ipynb) — attention-based sequence processing.
